**Calibração MACI (Agregação Multiplicativa)**

Calcula o resultado estatístico final

**Agregação Multiplicativa (Estado da Arte - MACI)**

Uma técnica recente (2024/2025) chamada Multi-LLM Adaptive Conformal Inference (MACI) propõe modelar a factualidade como o produto dos scores das sentenças ($s_{doc} = 1 - \prod (1 - s_i)$). Isso é mais robusto e evita que o sistema seja sensível demais a um único erro de estimativa de uma frase.4. Ajuste no Roteiro de ImplementaçãoPara o seu próximo script, vamos organizar os dados assim:Agrupar por id_bo: Calcular a média e o máximo dos scores de cada boletim.Reportar Factualidade Agregada: No artigo, você apresentará uma tabela com:% de Sentenças Factuais (Finitas).% de Relatórios Íntegros (Document-level).

In [4]:
import pandas as pd
import numpy as np

In [5]:
def calcular_maci_conformal(csv_auditado, alpha=0.05):
    df = pd.read_csv(csv_auditado)
    
    # 1. Agregação Multiplicativa (Frequência de Factualidade do Documento)
    # Probabilidade de ser fiel = 1 - s_i
    df['prob_fiel'] = 1 - df['non_conformity_score']
    
    # Produto das probabilidades por BO
    df_docs = df.groupby('id_bo')['prob_fiel'].prod().reset_index()
    
    # s_doc = 1 - Produto(prob_fiel)
    df_docs['s_doc'] = 1 - df_docs['prob_fiel']
    
    # 2. Particionamento
    df_calib = df_docs.sample(frac=0.5, random_state=42)
    df_test = df_docs.drop(df_calib.index)
    
    n = len(df_calib)
    scores_calib = df_calib['s_doc'].values
    
    # 3. Cálculo do Limiar Conformal q_hat
    # Fórmula rigorosa de Split Conformal: (n+1)(1-alpha)/n
    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_hat = np.quantile(scores_calib, q_level, method='higher')
    
    # 4. Avaliação
    df_test['aprovado'] = df_test['s_doc'] <= q_hat
    taxa_aprovacao = df_test['aprovado'].mean()
    erro_residual = df_test[df_test['aprovado'] == True]['s_doc'].mean()

    print(f"--- RESULTADOS FINAIS (MACI) ---")
    print(f"Limiar de Risco Aceitável (q_hat): {q_hat:.4f}")
    print(f"Taxa de Relatórios Íntegros (Aprovados): {taxa_aprovacao*100:.2f}%")
    print(f"Risco de Alucinação nos Aprovados: {erro_residual*100:.4f}%")
    
    df_test.to_csv("relatorios_finalizados_conformal.csv", index=False)
    return q_hat

In [6]:
calcular_maci_conformal("sentencas_auditadas_votos.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'sentencas_auditadas_votos.csv'